# Data Preparation

This notebook handles all data cleaning and saves the prepared dataset as
`data/processed/partially_selected_features.csv`.

**Steps:**
1. Drop irrelevant columns (identifiers, dates, empty)
2. Handle missing/invalid values with domain-specific reasoning

This notebook is separate from modeling so the data manipulation logic is clear and reproducible.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/insurance_claims.csv')
print(f"Original shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Original shape: (1000, 40)
Columns: ['months_as_customer', 'age', 'policy_number', 'policy_bind_date', 'policy_state', 'policy_csl', 'policy_deductable', 'policy_annual_premium', 'umbrella_limit', 'insured_zip', 'insured_sex', 'insured_education_level', 'insured_occupation', 'insured_hobbies', 'insured_relationship', 'capital-gains', 'capital-loss', 'incident_date', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state', 'incident_city', 'incident_location', 'incident_hour_of_the_day', 'number_of_vehicles_involved', 'property_damage', 'bodily_injuries', 'witnesses', 'police_report_available', 'total_claim_amount', 'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make', 'auto_model', 'auto_year', 'fraud_reported', '_c39']


## 1. Drop Irrelevant Columns

These columns are removed because they have no predictive value:
- `policy_number` — unique identifier, not a feature
- `policy_bind_date` — raw date string, not directly usable
- `insured_zip` — too many unique values, acts as an identifier
- `incident_location` — free-text-like, too specific and high cardinality
- `incident_date` — raw date string, not directly usable
- `_c39` — 100% missing values, empty column

In [2]:
cols_to_drop = [
    'policy_number',       # identifier - no predictive value
    'policy_bind_date',    # date string - not useful directly
    'insured_zip',         # too many unique values, identifier-like
    'incident_location',   # too specific, high cardinality
    'incident_date',       # date string - not useful directly
    '_c39',                # 100% missing values (empty column)
]

df = df.drop(columns=cols_to_drop)
print(f"Shape after dropping: {df.shape}")
print(f"Dropped: {cols_to_drop}")

Shape after dropping: (1000, 34)
Dropped: ['policy_number', 'policy_bind_date', 'insured_zip', 'incident_location', 'incident_date', '_c39']


## 2. Handle Missing / Invalid Values

Several columns use `?` as a placeholder for missing values, and `authorities_contacted` has
actual NaN values. Each is handled with domain-specific reasoning:

### a. `authorities_contacted` (91 missing, 9.1%)
The existing values are: Police, Fire, Ambulance, Other. A missing value logically means
**no authority was contacted** — if someone had been contacted, it would be recorded.
We assign the value `"None"` to represent this.

### b. `property_damage` (360 missing, 36.0%)
We assume that if property damage is not recorded, there was **no property damage** (beyond
the vehicle itself). We assign `"NO"`.

### c. `police_report_available` (343 missing, 34.3%)
Same reasoning as property_damage — if no police report is mentioned, we assume there is
**no police report available**. We assign `"NO"`.

### d. `collision_type` (178 missing, 17.8%)
Looking at the data, samples with `collision_type = ?` represent incidents where there was
no classical collision — theft, damage while parked, etc. We replace `?` with `"No collision"`.

In [3]:
# First, show current state of missing values
print("Before handling:")
print(f"  authorities_contacted NaN:    {df['authorities_contacted'].isna().sum()}")
print(f"  collision_type '?':           {(df['collision_type'] == '?').sum()}")
print(f"  property_damage '?':          {(df['property_damage'] == '?').sum()}")
print(f"  police_report_available '?':  {(df['police_report_available'] == '?').sum()}")

Before handling:
  authorities_contacted NaN:    91
  collision_type '?':           178
  property_damage '?':          360
  police_report_available '?':  343


In [4]:
# a. authorities_contacted: NaN -> "None" (no authority was contacted)
df['authorities_contacted'] = df['authorities_contacted'].fillna('None')
print("authorities_contacted — filled NaN with 'None'")
print(f"  Value counts:\n{df['authorities_contacted'].value_counts().to_string()}")
print()

authorities_contacted — filled NaN with 'None'
  Value counts:
authorities_contacted
Police       292
Fire         223
Other        198
Ambulance    196
None          91



In [5]:
# b. property_damage: '?' -> 'NO' (assume no property damage)
df['property_damage'] = df['property_damage'].replace('?', 'NO')
print("property_damage — replaced '?' with 'NO'")
print(f"  Value counts:\n{df['property_damage'].value_counts().to_string()}")
print()

property_damage — replaced '?' with 'NO'
  Value counts:
property_damage
NO     698
YES    302



In [6]:
# c. police_report_available: '?' -> 'NO' (assume no report available)
df['police_report_available'] = df['police_report_available'].replace('?', 'NO')
print("police_report_available — replaced '?' with 'NO'")
print(f"  Value counts:\n{df['police_report_available'].value_counts().to_string()}")
print()

police_report_available — replaced '?' with 'NO'
  Value counts:
police_report_available
NO     686
YES    314



In [7]:
# d. collision_type: '?' -> 'No collision' (theft, parked damage, etc.)
df['collision_type'] = df['collision_type'].replace('?', 'No collision')
print("collision_type — replaced '?' with 'No collision'")
print(f"  Value counts:\n{df['collision_type'].value_counts().to_string()}")
print()

collision_type — replaced '?' with 'No collision'
  Value counts:
collision_type
Rear Collision     292
Side Collision     276
Front Collision    254
No collision       178



In [8]:
# Verify: no missing values remain
remaining_missing = df.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

remaining_question = {}
for col in df.select_dtypes(include=['object']).columns:
    q = (df[col] == '?').sum()
    if q > 0:
        remaining_question[col] = q

if len(remaining_missing) == 0 and len(remaining_question) == 0:
    print("No missing values or '?' placeholders remain.")
else:
    if len(remaining_missing) > 0:
        print(f"Remaining NaN: {remaining_missing.to_dict()}")
    if len(remaining_question) > 0:
        print(f"Remaining '?': {remaining_question}")

print(f"\nFinal shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

No missing values or '?' placeholders remain.

Final shape: (1000, 34)
Columns (34): ['months_as_customer', 'age', 'policy_state', 'policy_csl', 'policy_deductable', 'policy_annual_premium', 'umbrella_limit', 'insured_sex', 'insured_education_level', 'insured_occupation', 'insured_hobbies', 'insured_relationship', 'capital-gains', 'capital-loss', 'incident_type', 'collision_type', 'incident_severity', 'authorities_contacted', 'incident_state', 'incident_city', 'incident_hour_of_the_day', 'number_of_vehicles_involved', 'property_damage', 'bodily_injuries', 'witnesses', 'police_report_available', 'total_claim_amount', 'injury_claim', 'property_claim', 'vehicle_claim', 'auto_make', 'auto_model', 'auto_year', 'fraud_reported']


/tmp/ipykernel_85946/2864470188.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


## 3. Save Prepared Dataset

In [9]:
output_path = '../data/processed/partially_selected_features.csv'
df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
print(f"Shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)

Saved to: ../data/processed/partially_selected_features.csv
Shape: (1000, 34)

Column types:
months_as_customer               int64
age                              int64
policy_state                       str
policy_csl                         str
policy_deductable                int64
policy_annual_premium          float64
umbrella_limit                   int64
insured_sex                        str
insured_education_level            str
insured_occupation                 str
insured_hobbies                    str
insured_relationship               str
capital-gains                    int64
capital-loss                     int64
incident_type                      str
collision_type                     str
incident_severity                  str
authorities_contacted              str
incident_state                     str
incident_city                      str
incident_hour_of_the_day         int64
number_of_vehicles_involved      int64
property_damage                    str
bodily_inj